<a href="https://colab.research.google.com/github/daniel789k/stock_forecast/blob/main/%E8%82%A1%E7%A5%A8%E9%A0%90%E6%B8%AC%E4%BB%8B%E9%9D%A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import tkinter as tk
import talib
from itertools import product
from pandastable import Table
from pandastable import images
from pandastable.dialogs import addButton

# 計算2線交點

In [ ]:
def cross_count(line1,line2):#計算交點函數
    x1=line1[0]#取四點座標
    y1=line1[1]
    x2=line1[2]
    y2=line1[3]

    x3=line2[0]
    y3=line2[1]
    x4=line2[2]
    y4=line2[3]

    k1=(y2-y1)*1.0/(x2-x1)#計算k1,由於點均爲整數，需要進行浮點數轉化
    b1=y1*1.0-x1*k1*1.0#整型轉浮點型是關鍵
    if (x4-x3)==0:#L2直線斜率不存在操作
        k2=None
        b2=0
    else:
        k2=(y4-y3)*1.0/(x4-x3)#斜率存在操作
        b2=y3*1.0-x3*k2*1.0
    if k2==None:
        x=x3
    else:
        x=(b2-b1)*1.0/(k1-k2)
    y=k1*x*1.0+b1*1.0
    return y

# 找出各指標

In [ ]:
def find_indicators():
    # KD
    All_K = []
    All_D = []

    #第一天KD值
    today_RSV = (float(stock_data[now_search[-1]][0][8:9]['close']) - min(list(stock_data[now_search[-1]][0][0:9]['min']))
                ) / (max(list(stock_data[now_search[-1]][0][0:9]['max'])) - min(list(stock_data[now_search[-1]][0][0:9]['min']))) * 100
    today_K = 2/3*50 + 1/3*today_RSV
    today_D = 2/3*50 + 1/3*today_K
    All_K.append(today_K)
    All_D.append(today_D)

    #第二到今日KD值
    a = 1
    while a <= stock_data[now_search[-1]][0].shape[0] - 9:
        if (max(list(stock_data[now_search[-1]][0][0+a:9+a]['max'])) - min(list(stock_data[now_search[-1]][0][0+a:9+a]['min']))) == 0:
            today_RSV = 0
        else:
            today_RSV = (float(stock_data[now_search[-1]][0][8+a:9+a]['close']) - min(list(stock_data[now_search[-1]][0][0+a:9+a]['min']))
                    ) / (max(list(stock_data[now_search[-1]][0][0+a:9+a]['max'])) - min(list(stock_data[now_search[-1]][0][0+a:9+a]['min']))) * 100
        today_K = 2/3*today_K + 1/3*today_RSV
        today_D = 2/3*today_D + 1/3*today_K
        All_K.append(today_K)
        All_D.append(today_D)
        a += 1

    while len(All_K) < len(stock_data[now_search[-1]][0]):
        All_K.insert(0,np.nan)
        All_D.insert(0,np.nan)

    stock_data[now_search[-1]][0]['All_K'] = All_K
    stock_data[now_search[-1]][0]['All_D'] = All_D


    # 布林通道
    stock_data[now_search[-1]][0]['upperband'], stock_data[now_search[-1]][0]['middleband'], stock_data[now_search[-1]][0]['lowerband'] = talib.BBANDS(stock_data[now_search[-1]][0]['close'],
                                                                    timeperiod=20,
                                                                    nbdevup=2, nbdevdn=2, matype=0)

    # RSI
    stock_data[now_search[-1]][0]['real'] = talib.RSI(stock_data[now_search[-1]][0]['close'], timeperiod=5)

# 各指標轉換評級數字

In [ ]:
# KD評級標準
KD_standard = [10,25,50,75,100]
KD_gold_number = [1,0.75,0.5,0.25,0.15]

KD_standard2 = [90,75,50,25,0]
KD_dead_number = [-1,-0.75,-0.5,-0.25,-0.15]

KD_nocross = [100,100]


# 布林通道評級標準
BB_standard = [-1,-1,0,1,1]


# RSI評級標準
RSI_standard = [2,100,0.01]

In [ ]:
def indicators_to_num():
    #KD值轉換評級數字
    KD_num = []
    All_K = [x for x in list(stock_data[now_search[-1]][0]['All_K']) if np.isnan(x) == False]
    All_D = [x for x in list(stock_data[now_search[-1]][0]['All_D']) if np.isnan(x) == False]
    cross = 0
    a = 0
    while a <= len(All_D)-1:
        KD_gap = All_K[a] - All_D[a]
        last_KD_gap = All_K[a-1] - All_D[a-1]
        # 發生交叉
        if KD_gap == 0 or (KD_gap > 0 and last_KD_gap < 0) or (KD_gap < 0 and last_KD_gap > 0):
            K_line = [0,All_K[a-1],1,All_K[a]]
            D_line = [0,All_D[a-1],1,All_D[a]]
            cross_point = cross_count(K_line, D_line)
            # 交叉前天K小於D:黃金交叉
            if last_KD_gap < 0:
                if cross_point < KD_standard[0]:
                    num  = KD_gold_number[0]
                elif KD_standard[0] <= cross_point < KD_standard[1]:
                    num = KD_gold_number[1]
                elif KD_standard[1] <= cross_point < KD_standard[2]:
                    num = KD_gold_number[2]
                elif KD_standard[2] <= cross_point < KD_standard[3]:
                    num = KD_gold_number[3]
                elif KD_standard[3] <= cross_point < KD_standard[4]:
                    num = KD_gold_number[4]
                cross = num
            # 交叉前天K大於D:死亡交叉
            elif last_KD_gap > 0:
                if cross_point > KD_standard2[0]:
                    num  = KD_dead_number[0]
                elif KD_standard2[1] < cross_point <= KD_standard2[0]:
                    num = KD_dead_number[1]
                elif KD_standard2[2] < cross_point <= KD_standard2[1]:
                    num = KD_dead_number[2]
                elif KD_standard2[3] < cross_point <= KD_standard2[2]:
                    num = KD_dead_number[3]
                elif KD_standard2[4] < cross_point <= KD_standard2[3]:
                    num = KD_dead_number[4]
                cross = num
        # 沒發生交叉
        elif (KD_gap > 0 and last_KD_gap > 0) or (KD_gap < 0 and last_KD_gap < 0):
            num = cross * ((KD_nocross[0]-abs(KD_gap)) /KD_nocross[1])

        KD_num.append(num)
        a += 1

    while len(KD_num) < len(stock_data[now_search[-1]][0]):
        KD_num.insert(0,np.nan)

    stock_data[now_search[-1]][0]['KD_num'] = KD_num

    # 布林通道轉換評級數字
    BB_num = []
    BB_up = [x for x in list(stock_data[now_search[-1]][0]['upperband']) if np.isnan(x) == False]
    BB_middle = [x for x in list(stock_data[now_search[-1]][0]['middleband']) if np.isnan(x) == False]
    BB_down = [x for x in list(stock_data[now_search[-1]][0]['lowerband']) if np.isnan(x) == False]

    a = 0
    while a <= len(BB_up)-1:
        today_close = float(stock_data[now_search[-1]][0][19+a:20+a]['close'])
        if today_close >= BB_up[a]:
            today_num = BB_standard[0]
        elif BB_middle[a] < today_close < BB_up[a]:
            today_num = ((today_close-BB_middle[a]) / (BB_up[a]-BB_middle[a])) * BB_standard[1]
        elif today_close == BB_middle[a]:
            today_num = BB_standard[2]
        elif BB_down[a] < today_close < BB_middle[a]:
            today_num = ((BB_middle[a]-today_close) / (BB_middle[a]-BB_down[a])) * BB_standard[3]
        elif today_close <= BB_down[a]:
            today_num = BB_standard[4]
        BB_num.append(today_num)
        a+=1

    while len(BB_num) < len(stock_data[now_search[-1]][0]):
        BB_num.insert(0,np.nan)

    stock_data[now_search[-1]][0]['BB_num'] = BB_num


    #RSI轉換評級數字
    RSI_num = []
    All_RSI = [x for x in list(stock_data[now_search[-1]][0]['real']) if np.isnan(x) == False]

    a = 0
    while a <= len(All_RSI)-1:
        num = (All_RSI[a]*RSI_standard[0] - RSI_standard[1]) * RSI_standard[2]
        RSI_num.append(num)
        a += 1

    while len(RSI_num) < len(stock_data[now_search[-1]][0]):
        RSI_num.insert(0,np.nan)
    stock_data[now_search[-1]][0]['RSI_num'] = RSI_num


# 回測

In [ ]:
def Buy_sell_Backtesting():
    test = list(product('01','01','01'))
    Buy_sell_data = pd.DataFrame()
    BB_num = [x for x in list(stock_data[now_search[-1]][0]['BB_num']) if np.isnan(x) == False]
    RSI_num = [x for x in list(stock_data[now_search[-1]][0]['RSI_num']) if np.isnan(x) == False]
    KD_num = [x for x in list(stock_data[now_search[-1]][0]['KD_num']) if np.isnan(x) == False]

    f = 1
    while f <= 5:
        e = -0.3
        while e >= -2:
            d = 0.3
            while d <= 2:
                c = 0
                while c <= len(test)-1:
                    #採用評級數字加總
                    Fin_num = []

                    b = -1825
                    while b <= -1:
                        num = 0
                        if test[c][0] == '1':
                            num += BB_num[b]
                        if test[c][1] == '1':
                            num += RSI_num[b]
                        if test[c][2] == '1':
                            num += KD_num[b]
                        Fin_num.append(num)
                        b += 1

                    # 買入持有策略
                    cost_buy = 0
                    earn_sell = 0
                    handling_fee = 0
                    handling_percent = 0.1425 * 0.01
                    tax_payment = 0
                    tax_percent = 0.3 * 0.01
                    buy_acount = 0
                    sell_acount = 0
                    hold = 0
                    hold_value = 0
                    buy_limit = d
                    sell_limit = e

                    DF = pd.DataFrame()

                    a = -365*f
                    while a <= -3:
                        if Fin_num[a] <= sell_limit and hold != 0:
                            earn_sell += float(stock_data[now_search[-1]][0][a+1:a+2]['close'])*1000
                            handling_fee += float(stock_data[now_search[-1]][0][a+1:a+2]['close'])*1000 * handling_percent
                            tax_payment += float(stock_data[now_search[-1]][0][a+1:a+2]['close'])*1000 * tax_percent
                            sell_DF = stock_data[now_search[-1]][0][a+1:a+2]
                            sell_DF.insert(1,'position','sell')
                            DF = DF.append(sell_DF,ignore_index=True)
                            hold -= 1
                            sell_acount += 1
                        elif Fin_num[a] >= buy_limit and hold == 0:
                            cost_buy += float(stock_data[now_search[-1]][0][a+1:a+2]['close'])*1000
                            handling_fee += float(stock_data[now_search[-1]][0][a+1:a+2]['close'])*1000 * handling_percent
                            buy_DF = stock_data[now_search[-1]][0][a+1:a+2]
                            buy_DF.insert(1,'position','buy')
                            DF = DF.append(buy_DF,ignore_index=True)
                            hold += 1
                            buy_acount += 1
                        a += 1

                    hold_value = hold * float(stock_data[now_search[-1]][0][a+1:]['close'])*1000

                    if cost_buy+handling_fee+tax_payment == 0:
                        Rate_Return = 0
                    else:
                        Rate_Return =  round((hold_value+earn_sell)/(cost_buy+handling_fee+tax_payment),2)

                    Buy_sell_data=Buy_sell_data.append({'buy_acount':buy_acount ,'sell_acount':sell_acount ,'hold':hold ,
                                                          'BB_use':test[c][0],'RSI_use':test[c][1],'KD_use':test[c][2],
                                                        'Buy_limit':buy_limit,'Sell_limit':sell_limit,'Test_year':f,
                                                         'rate_of_return' : Rate_Return}, ignore_index=True)
                    c += 1

                d += 0.1
            e -= 0.1
        f += 1
    Buy_sell_data = Buy_sell_data.sort_values(by=['rate_of_return'], ascending=False)
    return Buy_sell_data

In [ ]:
def Buy_hold_Backtesting():
    test = list(product('01','01','01'))
    Buy_hold_data = pd.DataFrame()
    BB_num = [x for x in list(stock_data[now_search[-1]][0]['BB_num']) if np.isnan(x) == False]
    RSI_num = [x for x in list(stock_data[now_search[-1]][0]['RSI_num']) if np.isnan(x) == False]
    KD_num = [x for x in list(stock_data[now_search[-1]][0]['KD_num']) if np.isnan(x) == False]

    e = 1
    while e <= 5:
        d = 0.3
        while d <= 2:
            c = 0
            while c <= len(test)-1:
                #採用評級數字加總
                Fin_num = []

                b = -1825
                while b <= -1:
                    num = 0
                    if test[c][0] == '1':
                        num += BB_num[b]
                    if test[c][1] == '1':
                        num += RSI_num[b]
                    if test[c][2] == '1':
                        num += KD_num[b]
                    Fin_num.append(num)
                    b += 1

                # 買入持有策略
                cost_buy = 0
                earn_sell = 0
                handling_fee = 0
                handling_percent = 0.1425 * 0.01
                tax_payment = 0
                tax_percent = 0.3 * 0.01
                buy_acount = 0
                sell_acount = 0
                hold = 0
                hold_value = 0
                buy_limit = d

                DF = pd.DataFrame()

                a = -365*e
                while a <= -3:
                    if Fin_num[a] >= buy_limit:
                        cost_buy += float(stock_data[now_search[-1]][0][a+1:a+2]['close'])*1000
                        handling_fee += float(stock_data[now_search[-1]][0][a+1:a+2]['close'])*1000 * handling_percent
                        buy_DF = stock_data[now_search[-1]][0][a+1:a+2]
                        buy_DF.insert(buy_DF.shape[1],'position','buy')
                        DF = DF.append(buy_DF,ignore_index=True)
                        hold += 1
                        buy_acount += 1
                    a += 1

                hold_value = hold * float(stock_data[now_search[-1]][0][a+1:]['close'])*1000

                if cost_buy+handling_fee+tax_payment == 0:
                    Rate_Return = 0
                else:
                    Rate_Return =  round((hold_value+earn_sell)/(cost_buy+handling_fee+tax_payment),2)

                Buy_hold_data=Buy_hold_data.append({'buy_acount':buy_acount ,'sell_acount':sell_acount ,'hold':hold ,
                                                      'BB_use':test[c][0],'RSI_use':test[c][1],'KD_use':test[c][2],
                                                    'Buy_limit':buy_limit,'Test_year':e,
                                                      'rate_of_return' : Rate_Return} , ignore_index=True)
                c += 1

            d += 0.1
        e += 1
    Buy_hold_data = Buy_hold_data.sort_values(by=['rate_of_return'], ascending=False)
    return Buy_hold_data

# 圖形化顯示

In [ ]:
stock_data = {}
stock_record = {}
#stock_graph = {}
testDF = []
now_search = []

In [ ]:
All_classification[24]

['2314',
 '2321',
 '2332',
 '2345',
 '2412',
 '2419',
 '2439',
 '2444',
 '2450',
 '2455',
 '2485',
 '2498',
 '3025',
 '3027',
 '3045',
 '3047',
 '3062',
 '3138',
 '3311',
 '3380',
 '3419',
 '3596',
 '3669',
 '3682',
 '3694',
 '3704',
 '4904',
 '4906',
 '4977',
 '5388',
 '6136',
 '6142',
 '6152',
 '6216',
 '6285',
 '6416',
 '6426',
 '6442',
 '6674',
 '6792',
 '8011',
 '8101']

In [ ]:
# 各類股股票代號
classificationList = ["0050成分股","金融股",'水泥股','食品股','塑膠股','紡織纖維股','電機機械股','電器電纜股','玻璃陶瓷股',
                     '造紙股','鋼鐵股','橡膠股','汽車股','建材營造股','航運股','觀光股','貿易百貨股','其他股','化學股','生技醫療股',
                     '油電燃氣股','半導體股','電腦及周邊設備股','光電股','通信網路股','電子零組件股','電子通路股','資訊服務股',
                     '其他電子股']

All_classification  = []

# 0050成分股
stock_0050 = ['2330','2454','2317','2303','2308','2881','2882','1303','2891','1301'
                ,'2412','2002','2886','2884','2603','3711','5871','1216','3037','2883'
                ,'2885','2892','5880','1101','1326','2357','3034','2382','2880','2887'
                ,'6415','2379','2609','2327','2615','3008','2207','2409','3045','5876'
                ,'2395','1590','2912','6505','2801','4904','8046','9910','2408','8454']
All_classification.append(stock_0050)

# 金融股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=17&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_finance = list(df)
All_classification.append(stock_finance)

# 水泥股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=01&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_cement = list(df)
All_classification.append(stock_cement)

# 食品股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=02&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_food = list(df)
All_classification.append(stock_food)

# 塑膠股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=03&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_plastic = list(df)
All_classification.append(stock_plastic)

# 紡織纖維股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=04&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_textile = list(df)
All_classification.append(stock_textile)

# 電機機械股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=05&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_motor = list(df)
All_classification.append(stock_motor)

# 電器電纜股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=06&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_cable = list(df)
All_classification.append(stock_cable)

# 玻璃陶瓷股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=08&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_glass = list(df)
All_classification.append(stock_glass)

# 造紙股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=09&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_Paper = list(df)
All_classification.append(stock_Paper)

# 鋼鐵股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=10&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_steel = list(df)
All_classification.append(stock_steel)

# 橡膠股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=11&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_rubber = list(df)
All_classification.append(stock_rubber)

# 汽車股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=12&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_car = list(df)
All_classification.append(stock_car)

# 建材營造股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=14&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_building = list(df)
All_classification.append(stock_building)

# 航運股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=15&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_Shipping = list(df)
All_classification.append(stock_Shipping)

# 觀光股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=16&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_sightseeing = list(df)
All_classification.append(stock_sightseeing)

# 貿易百貨股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=18&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_department = list(df)
All_classification.append(stock_department)

# 其他股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=20&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_other = list(df)
All_classification.append(stock_other)

# 化學股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=21&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_Chemical = list(df)
All_classification.append(stock_Chemical)

# 生技醫療股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=22&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_Medical = list(df)
All_classification.append(stock_Medical)

# 油電燃氣股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=23&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_Oil = list(df)
All_classification.append(stock_Oil)

# 半導體股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=24&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_semiconductor = list(df)
All_classification.append(stock_semiconductor)

# 電腦及周邊設備股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=25&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_computer = list(df)
All_classification.append(stock_computer)

# 光電股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=26&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_Light = list(df)
All_classification.append(stock_Light)

# 通信網路股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=27&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_network = list(df)
All_classification.append(stock_network)

# 電子零組件股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=28&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_Components = list(df)
All_classification.append(stock_Components)

# 電子通路股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=29&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_path = list(df)
All_classification.append(stock_path)

# 資訊服務股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=29&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_News = list(df)
All_classification.append(stock_News)

# 其他電子股
result = requests.get("https://isin.twse.com.tw/isin/class_main.jsp?owncode=&stockname=&isincode=&market=1&issuetype=&industry_code=31&Page=1&chklike=Y")
df = pd.read_html(result.text)[0][2][1:]
stock_electronic = list(df)
All_classification.append(stock_electronic)



In [ ]:
# 功能製作

# 選單功能
def onOpen():
    indexlabel2.grid()
    indexlabel3.grid()
    BuySelllabel.grid()
    BuySelllabel2.grid()
    BuySelllabel3.grid()
def onClose():
    indexlabel2.grid_remove()
    indexlabel3.grid_remove()
    BuySelllabel.grid_remove()
    BuySelllabel2.grid_remove()
    BuySelllabel3.grid_remove()



def KD_norm():
    def KD_norm_check():
        KD_standard[0] = int(KDentry.get())
        KD_standard[1] = int(KDentry2.get())
        KD_standard[2] = int(KDentry3.get())
        KD_standard[3] = int(KDentry4.get())
        KD_standard[4] = int(KDentry5.get())
        KD_gold_number[0] = float(KD_number_entry.get())
        KD_gold_number[1] = float(KD_number_entry2.get())
        KD_gold_number[2] = float(KD_number_entry3.get())
        KD_gold_number[3] = float(KD_number_entry4.get())
        KD_gold_number[4] = float(KD_number_entry5.get())
        KD_standard2[0] = int(KDentry_dead.get())
        KD_standard2[1] = int(KDentry_dead2.get())
        KD_standard2[2] = int(KDentry_dead3.get())
        KD_standard2[3] = int(KDentry_dead4.get())
        KD_standard2[4] = int(KDentry_dead5.get())
        KD_dead_number[0] = float(KD_deadnumber_entry.get())
        KD_dead_number[1] = float(KD_deadnumber_entry2.get())
        KD_dead_number[2] = float(KD_deadnumber_entry3.get())
        KD_dead_number[3] = float(KD_deadnumber_entry4.get())
        KD_dead_number[4] = float(KD_deadnumber_entry5.get())

        KD_nocross[0] = int(nocross_entry.get())
        KD_nocross[1] = int(nocross_entry2.get())

        stock_data.clear()
        stock_record.clear()
        newWindow.destroy()

    newWindow = tk.Toplevel(app)
    newWindow.title(f'KD指標評級數字設定')
    newWindow.geometry('500x430')

    cross_truelabel = tk.Label(newWindow, text='發生交叉')
    cross_truelabel.grid(row=0, column=1, columnspan=2)

    cross_goldlabel = tk.Label(newWindow, text='黃金交叉')
    cross_goldlabel.grid(row=1, column=2, columnspan=2)

    gold_pointlabel = tk.Label(newWindow, text='交叉位置')
    gold_pointlabel.grid(row=2, column=3, columnspan=2)

    gold_smalllabel = tk.Label(newWindow, text='小於')
    gold_smalllabel.grid(row=2, column=5)

    KDentry = tk.Entry(newWindow, width=5)
    KDentry.insert(0, str(KD_standard[0]))
    KDentry.grid(row=2, column=6, sticky=tk.W)

    KDentry2 = tk.Entry(newWindow, width=5)
    KDentry2.insert(0, str(KD_standard[1]))
    KDentry2.grid(row=3, column=6, sticky=tk.W)

    KDentry3 = tk.Entry(newWindow, width=5)
    KDentry3.insert(0, str(KD_standard[2]))
    KDentry3.grid(row=4, column=6, sticky=tk.W)

    KDentry4 = tk.Entry(newWindow, width=5)
    KDentry4.insert(0, str(KD_standard[3]))
    KDentry4.grid(row=5, column=6, sticky=tk.W)

    KDentry5 = tk.Entry(newWindow, width=5)
    KDentry5.insert(0, str(KD_standard[4]))
    KDentry5.grid(row=6, column=6, sticky=tk.W)

    gold_numberlabel = tk.Label(newWindow, text='評級數字')
    gold_numberlabel.grid(row=2, column=7, columnspan=2)
    gold_remindlabel = tk.Label(newWindow, text='(須為1~-1)')
    gold_remindlabel.grid(row=3, column=7, columnspan=2)

    KD_number_entry = tk.Entry(newWindow, width=5)
    KD_number_entry.insert(0, str(KD_gold_number[0]))
    KD_number_entry.grid(row=2, column=9, sticky=tk.W)

    KD_number_entry2 = tk.Entry(newWindow, width=5)
    KD_number_entry2.insert(0, str(KD_gold_number[1]))
    KD_number_entry2.grid(row=3, column=9, sticky=tk.W)

    KD_number_entry3 = tk.Entry(newWindow, width=5)
    KD_number_entry3.insert(0, str(KD_gold_number[2]))
    KD_number_entry3.grid(row=4, column=9, sticky=tk.W)

    KD_number_entry4 = tk.Entry(newWindow, width=5)
    KD_number_entry4.insert(0, str(KD_gold_number[3]))
    KD_number_entry4.grid(row=5, column=9, sticky=tk.W)

    KD_number_entry5 = tk.Entry(newWindow, width=5)
    KD_number_entry5.insert(0, str(KD_gold_number[4]))
    KD_number_entry5.grid(row=6, column=9, sticky=tk.W)

    null = tk.Label(newWindow, text='')
    null.grid(row=7, column=0)


    cross_deadlabel = tk.Label(newWindow, text='死亡交叉')
    cross_deadlabel.grid(row=8, column=2, columnspan=2)

    dead_pointlabel = tk.Label(newWindow, text='交叉位置')
    dead_pointlabel.grid(row=9, column=3, columnspan=2)

    dead_biglabel = tk.Label(newWindow, text='大於')
    dead_biglabel.grid(row=9, column=5)

    KDentry_dead = tk.Entry(newWindow, width=5)
    KDentry_dead.insert(0, str(KD_standard2[0]))
    KDentry_dead.grid(row=9, column=6, sticky=tk.W)

    KDentry_dead2 = tk.Entry(newWindow, width=5)
    KDentry_dead2.insert(0, str(KD_standard2[1]))
    KDentry_dead2.grid(row=10, column=6, sticky=tk.W)

    KDentry_dead3 = tk.Entry(newWindow, width=5)
    KDentry_dead3.insert(0, str(KD_standard2[2]))
    KDentry_dead3.grid(row=11, column=6, sticky=tk.W)

    KDentry_dead4 = tk.Entry(newWindow, width=5)
    KDentry_dead4.insert(0, str(KD_standard2[3]))
    KDentry_dead4.grid(row=12, column=6, sticky=tk.W)

    KDentry_dead5 = tk.Entry(newWindow, width=5)
    KDentry_dead5.insert(0, str(KD_standard2[4]))
    KDentry_dead5.grid(row=13, column=6, sticky=tk.W)

    dead_numberlabel = tk.Label(newWindow, text='評級數字')
    dead_numberlabel.grid(row=9, column=7, columnspan=2)

    KD_deadnumber_entry = tk.Entry(newWindow, width=5)
    KD_deadnumber_entry.insert(0, str(KD_dead_number[0]))
    KD_deadnumber_entry.grid(row=9, column=9, sticky=tk.W)

    KD_deadnumber_entry2 = tk.Entry(newWindow, width=5)
    KD_deadnumber_entry2.insert(0, str(KD_dead_number[1]))
    KD_deadnumber_entry2.grid(row=10, column=9, sticky=tk.W)

    KD_deadnumber_entry3 = tk.Entry(newWindow, width=5)
    KD_deadnumber_entry3.insert(0, str(KD_dead_number[2]))
    KD_deadnumber_entry3.grid(row=11, column=9, sticky=tk.W)

    KD_deadnumber_entry4 = tk.Entry(newWindow, width=5)
    KD_deadnumber_entry4.insert(0, str(KD_dead_number[3]))
    KD_deadnumber_entry4.grid(row=12, column=9, sticky=tk.W)

    KD_deadnumber_entry5 = tk.Entry(newWindow, width=5)
    KD_deadnumber_entry5.insert(0, str(KD_dead_number[4]))
    KD_deadnumber_entry5.grid(row=13, column=9, sticky=tk.W)

    null2 = tk.Label(newWindow, text='')
    null2.grid(row=14, column=0)

    nocross_falselabel = tk.Label(newWindow, text='未發生交叉')
    nocross_falselabel.grid(row=15, column=1, columnspan=2)

    nocross_currentlabel = tk.Label(newWindow, text='最近一次交叉點 x (')
    nocross_currentlabel.grid(row=16, column=2, columnspan=3)

    nocross_entry = tk.Entry(newWindow, width=5)
    nocross_entry.insert(0, str(KD_nocross[0]))
    nocross_entry.grid(row=16, column=5, sticky=tk.W)

    nocross_gaplabel = tk.Label(newWindow, text='減 KD差距 )  除')
    nocross_gaplabel.grid(row=16, column=6, columnspan=2)

    nocross_entry2 = tk.Entry(newWindow, width=5)
    nocross_entry2.insert(0, str(KD_nocross[1]))
    nocross_entry2.grid(row=16, column=8, sticky=tk.W)

    nocross_remindlabel = tk.Label(newWindow, text='(須為整數)')
    nocross_remindlabel.grid(row=16, column=9, columnspan=2)

    null3 = tk.Label(newWindow, text='')
    null3.grid(row=17, column=0)


    KD_check = tk.Button(newWindow, text='確認', command=KD_norm_check)
    KD_check.grid(row=18, column=9, sticky=tk.W+tk.E, columnspan=2)


def BB_norm():
    def BB_norm_check():
        BB_standard[0] = float(BB_entry.get())
        BB_standard[1] = float(BB_entry2.get())
        BB_standard[2] = float(BB_entry3.get())
        BB_standard[3] = float(BB_entry4.get())
        BB_standard[4] = float(BB_entry5.get())

        stock_data.clear()
        stock_record.clear()
        newWindow.destroy()

    newWindow = tk.Toplevel(app)
    newWindow.title(f'布林通道指標評級數字設定')
    newWindow.geometry('500x280')

    stock_pricelabel = tk.Label(newWindow, text='股價')
    stock_pricelabel.grid(row=0, column=1, columnspan=2, pady=2, padx=20)

    over_uplabel = tk.Label(newWindow, text='突破上限')
    over_uplabel.grid(row=1, column=1, columnspan=2, sticky=tk.W, pady=2, padx=20)

    up_middlelabel = tk.Label(newWindow, text='上~中')
    up_middlelabel.grid(row=2, column=1, columnspan=2, sticky=tk.W, pady=2, padx=20)

    stock_middlelabel = tk.Label(newWindow, text='位於中線')
    stock_middlelabel.grid(row=3, column=1, columnspan=2, sticky=tk.W, pady=2, padx=20)

    middle_downlabel = tk.Label(newWindow, text='中~下')
    middle_downlabel.grid(row=4, column=1, columnspan=2, sticky=tk.W, pady=2, padx=20)

    over_downlabel = tk.Label(newWindow, text='突破下線')
    over_downlabel.grid(row=5, column=1, columnspan=2, sticky=tk.W, pady=2, padx=20)


    BB_numberlabel = tk.Label(newWindow, text='評級數字')
    BB_numberlabel.grid(row=0, column=3, columnspan=7, sticky=tk.W)

    BB_entry = tk.Entry(newWindow, width=5)
    BB_entry.insert(0, str(BB_standard[0]))
    BB_entry.grid(row=1, column=3, sticky=tk.W)

    BB_entry2 = tk.Entry(newWindow, width=5)
    BB_entry2.insert(0, str(BB_standard[1]))
    BB_entry2.grid(row=2, column=3, sticky=tk.W)

    BB_entry3 = tk.Entry(newWindow, width=5)
    BB_entry3.insert(0, str(BB_standard[2]))
    BB_entry3.grid(row=3, column=3, sticky=tk.W)

    BB_entry4 = tk.Entry(newWindow, width=5)
    BB_entry4.insert(0, str(BB_standard[3]))
    BB_entry4.grid(row=4, column=3, sticky=tk.W)

    BB_entry5 = tk.Entry(newWindow, width=5)
    BB_entry5.insert(0, str(BB_standard[4]))
    BB_entry5.grid(row=5, column=3, sticky=tk.W)


    upmiddle_captionlabel = tk.Label(newWindow, text=' * (今日股價~中線距離) / (上線~中線距離)')
    upmiddle_captionlabel.grid(row=2, column=4, columnspan=6)

    middledown_captionlabel = tk.Label(newWindow, text=' * (中線~今日股價距離) / (中線~下線距離)')
    middledown_captionlabel.grid(row=4, column=4, columnspan=6)

    BB_check = tk.Button(newWindow, text='確認', command=BB_norm_check)
    BB_check.grid(row=6, column=8, sticky=tk.W+tk.E, columnspan=2)


def RSI_norm():
    def RSI_norm_check():
        RSI_standard[0] = float(RSI_entry.get())
        RSI_standard[1] = float(RSI_entry2.get())
        RSI_standard[2] = float(RSI_entry3.get())

        stock_data.clear()
        stock_record.clear()
        newWindow.destroy()

    newWindow = tk.Toplevel(app)
    newWindow.title(f'布林通道指標評級數字設定')
    newWindow.geometry('350x150')

    RSI_numberlabel = tk.Label(newWindow, text='評級數字 : ')
    RSI_numberlabel.grid(row=0, column=1, columnspan=2)

    today_RSIlabel = tk.Label(newWindow, text='(今日RSI值 * ')
    today_RSIlabel.grid(row=0, column=3, columnspan=2)

    RSI_entry = tk.Entry(newWindow, width=5)
    RSI_entry.insert(0, str(RSI_standard[0]))
    RSI_entry.grid(row=0, column=5, sticky=tk.W)

    minus_label = tk.Label(newWindow, text=' - ')
    minus_label.grid(row=0, column=6)

    RSI_entry2 = tk.Entry(newWindow, width=5)
    RSI_entry2.insert(0, str(RSI_standard[1]))
    RSI_entry2.grid(row=0, column=7, sticky=tk.W)

    star_label = tk.Label(newWindow, text=') * ')
    star_label.grid(row=0, column=8)

    RSI_entry3 = tk.Entry(newWindow, width=5)
    RSI_entry3.insert(0, str(RSI_standard[2]))
    RSI_entry3.grid(row=0, column=9, sticky=tk.W)

    null = tk.Label(newWindow, text='')
    null.grid(row=1, column=0)

    RSI_check = tk.Button(newWindow, text='確認', command=RSI_norm_check)
    RSI_check.grid(row=2, column=9, sticky=tk.W+tk.E, columnspan=2)



# 指標勾選功能
def KD_Buy_Sell():
    if KD_Value.get() == True:
        BuySellFrom.append(1)
        buyScale['from_'] = -sum(BuySellFrom)
        buyScale['to'] = sum(BuySellFrom)
        sellScale['from_'] = -sum(BuySellFrom)
        sellScale['to'] = sum(BuySellFrom)
    if KD_Value.get() == False:
        BuySellFrom.append(-1)
        buyScale['from_'] = -sum(BuySellFrom)
        buyScale['to'] = sum(BuySellFrom)
        sellScale['from_'] = -sum(BuySellFrom)
        sellScale['to'] = sum(BuySellFrom)

def BB_Buy_Sell():
    if BB_Value.get() == True:
        BuySellFrom.append(1)
        buyScale['from_'] = -sum(BuySellFrom)
        buyScale['to'] = sum(BuySellFrom)
        sellScale['from_'] = -sum(BuySellFrom)
        sellScale['to'] = sum(BuySellFrom)
    if BB_Value.get() == False:
        BuySellFrom.append(-1)
        buyScale['from_'] = -sum(BuySellFrom)
        buyScale['to'] = sum(BuySellFrom)
        sellScale['from_'] = -sum(BuySellFrom)
        sellScale['to'] = sum(BuySellFrom)

def RSI_Buy_Sell():
    if RSI_Value.get() == True:
        BuySellFrom.append(1)
        buyScale['from_'] = -sum(BuySellFrom)
        buyScale['to'] = sum(BuySellFrom)
        sellScale['from_'] = -sum(BuySellFrom)
        sellScale['to'] = sum(BuySellFrom)
    if RSI_Value.get() == False:
        BuySellFrom.append(-1)
        buyScale['from_'] = -sum(BuySellFrom)
        buyScale['to'] = sum(BuySellFrom)
        sellScale['from_'] = -sum(BuySellFrom)
        sellScale['to'] = sum(BuySellFrom)


def search():
    stock_num = myentry.get()
    if stock_num == '':
        lbl_1['text'] = '尚未輸入股票代號'
        return
    now_search.append(stock_num)

    if type(stock_data.get(stock_num)) != type(testDF):
        # 找股票資料
        today = datetime.date.today()
        today_10y_ago = today - datetime.timedelta(3650)

        url = "https://api.finmindtrade.com/api/v4/data"
        parameter = {
            "dataset": "TaiwanStockPrice",
            "data_id": stock_num,
            "start_date": str(today_10y_ago),
            "end_date": str(today),
            "token": "", # 參考登入，獲取金鑰
        }
        resp = requests.get(url, params=parameter)
        data = resp.json()
        data = pd.DataFrame(data["data"])
        stock_data[stock_num] = [data]
        find_indicators()
        indicators_to_num()


    if len(stock_data[stock_num]) < 3:
        Backtestingbutton.grid()
        Backtestinglabel2.grid()
        Backtestinglabel.grid_remove()
        buttonExample.grid_remove()
        buttonExample2.grid_remove()
    if len(stock_data[stock_num]) >= 3:
        Backtestingbutton.grid_remove()
        Backtestinglabel2.grid_remove()
        Backtestinglabel.grid()
        buttonExample.grid()
        buttonExample2.grid()

    # 找出指標list
    BB_num = [x for x in list(stock_data[now_search[-1]][0]['BB_num']) if np.isnan(x) == False]
    RSI_num = [x for x in list(stock_data[now_search[-1]][0]['RSI_num']) if np.isnan(x) == False]
    KD_num = [x for x in list(stock_data[now_search[-1]][0]['KD_num']) if np.isnan(x) == False]


    #採用評級數字加總
    Fin_num = []

    b = -len(BB_num)
    while b <= -1:
        num = 0
        if BB_Value.get() == True:
            num += BB_num[b]
        if RSI_Value.get() == True:
            num += RSI_num[b]
        if KD_Value.get() == True:
            num += KD_num[b]
        Fin_num.append(num)
        b += 1

    # 買入持有策略
    cost_buy = 0
    earn_sell = 0
    handling_fee = 0
    handling_percent = 0.1425 * 0.01
    tax_payment = 0
    tax_percent = 0.3 * 0.01
    buy_acount = 0
    sell_acount = 0
    hold = 0
    hold_value = 0
    buy_limit = float(buyScale.get())
    if radioValue.get() == 1:
        sell_limit = float(sellScale.get())

    DF = pd.DataFrame()

    if len(BB_num) < 365*int(timebox.get()):
        lbl_1['text'] = f'此股票尚未超過{timebox.get()}年'
        return

    if radioValue.get() == 0:
        c = -365*int(timebox.get())
        while c <= -3:
            if Fin_num[c] >= buy_limit:
                cost_buy += float(stock_data[now_search[-1]][0][c+1:c+2]['close'])*1000
                handling_fee += float(stock_data[now_search[-1]][0][c+1:c+2]['close'])*1000 * handling_percent
                buy_DF = stock_data[now_search[-1]][0][c+1:c+2]
                buy_DF.insert(1,'position','buy')
                DF = DF.append(buy_DF,ignore_index=True)
                hold += 1
                buy_acount += 1
            c += 1

    if radioValue.get() == 1:
        c = -365*int(timebox.get())
        while c <= -3:
            if Fin_num[c] <= sell_limit and hold != 0:
                earn_sell += float(stock_data[now_search[-1]][0][c+1:c+2]['close'])*1000
                handling_fee += float(stock_data[now_search[-1]][0][c+1:c+2]['close'])*1000 * handling_percent
                tax_payment += float(stock_data[now_search[-1]][0][c+1:c+2]['close'])*1000 * tax_percent
                sell_DF = stock_data[now_search[-1]][0][c+1:c+2]
                sell_DF.insert(1,'position','sell')
                DF = DF.append(sell_DF,ignore_index=True)
                hold -= 1
                sell_acount += 1
            elif Fin_num[c] >= buy_limit and hold == 0:
                cost_buy += float(stock_data[now_search[-1]][0][c+1:c+2]['close'])*1000
                handling_fee += float(stock_data[now_search[-1]][0][c+1:c+2]['close'])*1000 * handling_percent
                buy_DF = stock_data[now_search[-1]][0][c+1:c+2]
                buy_DF.insert(1,'position','buy')
                DF = DF.append(buy_DF,ignore_index=True)
                hold += 1
                buy_acount += 1
            c += 1

    long = []
    a = -1
    while len(long) <= 8:
        long.append(a)
        a -= 1

    if len(DF.columns) != 0:
        DF.drop(DF.columns[long], axis = 1,inplace=True)

    stock_record[stock_num] = DF

    hold_value = hold * float(stock_data[now_search[-1]][0][c+1:]['close'])*1000

    if (cost_buy+handling_fee+tax_payment) != 0:
        lbl_1['text'] = '最近'+timebox.get()+'年回測績效: '+str(round((hold_value+earn_sell)/(cost_buy+handling_fee+tax_payment),2))
    if (cost_buy+handling_fee+tax_payment) == 0:
        lbl_1['text'] = '最近'+timebox.get()+'年回測績效: 0'

    buylabel['text'] = '購買次數:'+str(buy_acount)
    selllabel['text'] = '出售次數:'+str(sell_acount)


def Backtesting():
    # 加入回測數據
    All_stock_data1 = stock_data[now_search[-1]][0]
    All_stock_data2 = Buy_hold_Backtesting()
    All_stock_data3 = Buy_sell_Backtesting()
    stock_data[now_search[-1]] = [All_stock_data1,All_stock_data2,All_stock_data3]

    Backtestingbutton.grid_remove()
    Backtestinglabel2.grid_remove()
    Backtestinglabel.grid()
    buttonExample.grid()
    buttonExample2.grid()


def createBacktesting():
    def my_callback():
        stock_data[now_search[-1]][1].to_excel(f'{now_search[-1]}買入持有詳細回測.xls')

    newWindow = tk.Toplevel(app)
    newWindow.title(f'{now_search[-1]}買入持有詳細回測')
    newWindow.geometry('830x400')

    toolbar = tk.Frame(newWindow)
    toolbar.pack()

    img1 = images.save()
    addButton(toolbar, 'Export Excel', my_callback, img1)

    frame = tk.Frame(newWindow)
    frame.pack(fill='both', expand=True)

    pt = Table(frame, dataframe=stock_data[now_search[-1]][1], showtoolbar=False, showstatusbar=False)
    pt.show()



def createBacktesting2():
    def my_callback():
        stock_data[now_search[-1]][2].to_excel(f'{now_search[-1]}短期買賣詳細回測.xls')

    newWindow = tk.Toplevel(app)
    newWindow.title(f'{now_search[-1]}短期買賣詳細回測')
    newWindow.geometry('900x400')

    toolbar = tk.Frame(newWindow)
    toolbar.pack()

    img1 = images.save()
    addButton(toolbar, 'Export Excel', my_callback, img1)

    frame = tk.Frame(newWindow)
    frame.pack(fill='both', expand=True)

    pt = Table(frame, dataframe=stock_data[now_search[-1]][2], showtoolbar=False, showstatusbar=False)
    pt.show()



def createRecord():
    def my_callback():
        stock_record[now_search[-1]].to_excel(f'{now_search[-1]}此次交易紀錄.xls')
    #pt.doExport(filename="test2.csv")
    newWindow = tk.Toplevel(app)
    newWindow.title(f'{now_search[-1]}此次交易紀錄')
    newWindow.geometry('1150x400')

    toolbar = tk.Frame(newWindow)
    toolbar.pack()

    img1 = images.save()
    addButton(toolbar, 'Export Excel', my_callback, img1)

    frame = tk.Frame(newWindow)
    frame.pack(fill='both', expand=True)

    pt = Table(frame, dataframe=stock_record[now_search[-1]], showtoolbar=False, showstatusbar=False)
    pt.show()


def createOption():
    a = 0
    while a <= len(classificationList)-1:
        if OptionWord.get() == classificationList[a]:
            OptionTest = All_classification[a]
            break
        a += 1

    Option_data = pd.DataFrame()
    q = 0
    while q <= len(OptionTest)-1:
        now_search.append(OptionTest[q])
        if type(stock_data.get(OptionTest[q])) != type(testDF):
            # 找股票資料
            today = datetime.date.today()
            today_10y_ago = today - datetime.timedelta(3650)

            url = "https://api.finmindtrade.com/api/v4/data"
            parameter = {
                "dataset": "TaiwanStockPrice",
                "data_id": OptionTest[q],
                "start_date": str(today_10y_ago),
                "end_date": str(today),
                "token": "", # 參考登入，獲取金鑰
            }
            resp = requests.get(url, params=parameter)
            data = resp.json()
            data = pd.DataFrame(data["data"])
            stock_data[OptionTest[q]] = [data]
            find_indicators()
            indicators_to_num()


        if len(stock_data[OptionTest[q]]) < 3:
            Backtestingbutton.grid()
            Backtestinglabel2.grid()
            Backtestinglabel.grid_remove()
            buttonExample.grid_remove()
            buttonExample2.grid_remove()
        if len(stock_data[OptionTest[q]]) >= 3:
            Backtestingbutton.grid_remove()
            Backtestinglabel2.grid_remove()
            Backtestinglabel.grid()
            buttonExample.grid()
            buttonExample2.grid()

        # 找出指標list
        BB_num = [x for x in list(stock_data[OptionTest[q]][0]['BB_num']) if np.isnan(x) == False]
        RSI_num = [x for x in list(stock_data[OptionTest[q]][0]['RSI_num']) if np.isnan(x) == False]
        KD_num = [x for x in list(stock_data[OptionTest[q]][0]['KD_num']) if np.isnan(x) == False]

        if len(BB_num) == 0:
            q += 1
            continue

        #採用評級數字加總
        Fin_num = []

        b = -len(BB_num)
        while b <= -1:
            num = 0
            if BB_Value.get() == True:
                num += BB_num[b]
            if RSI_Value.get() == True:
                num += RSI_num[b]
            if KD_Value.get() == True:
                num += KD_num[b]
            Fin_num.append(num)
            b += 1

        # 買入持有策略
        cost_buy = 0
        earn_sell = 0
        handling_fee = 0
        handling_percent = 0.1425 * 0.01
        tax_payment = 0
        tax_percent = 0.3 * 0.01
        buy_acount = 0
        sell_acount = 0
        hold = 0
        hold_value = 0
        buy_limit = float(buyScale.get())
        if radioValue.get() == 1:
            sell_limit = float(sellScale.get())

        DF = pd.DataFrame()

        if len(BB_num) < 365*int(timebox.get()):
            q += 1
            #lbl_1['text'] = f'此股票尚未超過{timebox.get()}年'
            continue

        if radioValue.get() == 0:
            c = -365*int(timebox.get())
            while c <= -3:
                if Fin_num[c] >= buy_limit:
                    cost_buy += float(stock_data[OptionTest[q]][0][c+1:c+2]['close'])*1000
                    handling_fee += float(stock_data[OptionTest[q]][0][c+1:c+2]['close'])*1000 * handling_percent
                    buy_DF = stock_data[OptionTest[q]][0][c+1:c+2]
                    buy_DF.insert(1,'position','buy')
                    DF = DF.append(buy_DF,ignore_index=True)
                    hold += 1
                    buy_acount += 1
                c += 1

        if radioValue.get() == 1:
            c = -365*int(timebox.get())
            while c <= -3:
                if Fin_num[c] <= sell_limit and hold != 0:
                    earn_sell += float(stock_data[OptionTest[q]][0][c+1:c+2]['close'])*1000
                    handling_fee += float(stock_data[OptionTest[q]][0][c+1:c+2]['close'])*1000 * handling_percent
                    tax_payment += float(stock_data[OptionTest[q]][0][c+1:c+2]['close'])*1000 * tax_percent
                    sell_DF = stock_data[OptionTest[q]][0][c+1:c+2]
                    sell_DF.insert(1,'position','sell')
                    DF = DF.append(sell_DF,ignore_index=True)
                    hold -= 1
                    sell_acount += 1
                elif Fin_num[c] >= buy_limit and hold == 0:
                    cost_buy += float(stock_data[OptionTest[q]][0][c+1:c+2]['close'])*1000
                    handling_fee += float(stock_data[OptionTest[q]][0][c+1:c+2]['close'])*1000 * handling_percent
                    buy_DF = stock_data[OptionTest[q]][0][c+1:c+2]
                    buy_DF.insert(1,'position','buy')
                    DF = DF.append(buy_DF,ignore_index=True)
                    hold += 1
                    buy_acount += 1
                c += 1

        long = []
        a = -1
        while len(long) <= 8:
            long.append(a)
            a -= 1

        if len(DF.columns) != 0:
            DF.drop(DF.columns[long], axis = 1,inplace=True)

        #stock_record[stock_num] = DF

        hold_value = hold * float(stock_data[OptionTest[q]][0][c+1:]['close'])*1000

        Option_data=Option_data.append({'stock_num' : OptionTest[q] ,'buy_acount':buy_acount ,
                                          'sell_acount':sell_acount ,'hold':hold ,
                                          'rate of return' : round((hold_value+earn_sell)/(cost_buy+handling_fee+tax_payment)
                                                            ,2)} , ignore_index=True)
        q += 1

    now_search.append(myentry.get())
    if now_search[-1] != '':
        if len(stock_data[now_search[-1]]) < 3:
            Backtestingbutton.grid()
            Backtestinglabel.grid_remove()
            buttonExample.grid_remove()
            buttonExample2.grid_remove()
        if len(stock_data[now_search[-1]]) >= 3:
            Backtestingbutton.grid_remove()
            Backtestinglabel.grid()
            buttonExample.grid()
            buttonExample2.grid()

    def my_callback():
        Option_data.to_excel(f'{OptionWord.get()}在此交易策略下之報酬率.xls')

    newWindow = tk.Toplevel(app)
    newWindow.title(f'{OptionWord.get()}在此交易策略下之報酬率')
    newWindow.geometry('600x400')

    toolbar = tk.Frame(newWindow)
    toolbar.pack()

    img1 = images.save()
    addButton(toolbar, 'Export Excel', my_callback, img1)

    frame = tk.Frame(newWindow)
    frame.pack(fill='both', expand=True)

    pt = Table(frame, dataframe=Option_data, showtoolbar=False, showstatusbar=False)
    pt.show()


In [ ]:
# 介面布局
app = tk.Tk()
app.title('主頁面')
app.geometry('380x600')


# 設定標籤
menubar = tk.Menu(app)

filemenu = tk.Menu(menubar)
filemenu.add_command(label="Open", command=onOpen)
filemenu.add_command(label="Close", command=onClose)

menubar.add_cascade(label="教學", menu=filemenu)

filemenu2 = tk.Menu(menubar)
filemenu2.add_command(label="KD指標", command=KD_norm)
filemenu2.add_command(label="布林通道", command=BB_norm)
filemenu2.add_command(label="RSI", command=RSI_norm)

menubar.add_cascade(label="指標設定", menu=filemenu2)

app.config(menu=menubar)

# 查詢股票代號
mylabel = tk.Label(app, text='股票代號:')
mylabel.grid(row=0, column=1, columnspan=2)
myentry = tk.Entry(app)
myentry.grid(row=0, column=3, sticky=tk.W, columnspan=3)

null = tk.Label(app, text='')
null.grid(row=1, column=0)

# 選擇要用指標
BuySellFrom = []
indexlabel = tk.Label(app, text='選擇想使用的回測指標')
indexlabel.grid(row=2, column=1, sticky=tk.W, columnspan=3)
KD_Value = tk.BooleanVar()
KD_Value.set(False)
BB_Value = tk.BooleanVar()
BB_Value.set(False)
RSI_Value = tk.BooleanVar()
RSI_Value.set(False)

KD_Bool = tk.Checkbutton(app, text='KD值', var=KD_Value, command=KD_Buy_Sell)
KD_Bool.grid(row=3, column=1, sticky=tk.W, columnspan=3)
BB_Bool = tk.Checkbutton(app, text='布林通道', var=BB_Value, command=BB_Buy_Sell)
BB_Bool.grid(row=4, column=1, sticky=tk.W, columnspan=3)
RSI_Bool = tk.Checkbutton(app, text='RSI', var=RSI_Value, command=RSI_Buy_Sell)
RSI_Bool.grid(row=5, column=1, sticky=tk.W, columnspan=3)

indexlabel2 = tk.Label(app, text='系統將根據各指標當日狀況訂出1~-1的評級數字')
indexlabel2.grid(row=6, column=1, sticky=tk.W, columnspan=7)
indexlabel3 = tk.Label(app, text='Ex : 突破布林下線時上漲機率高設為1，突破上限時為-1')
indexlabel3.grid(row=7, column=1, sticky=tk.W, columnspan=7)

null2 = tk.Label(app, text='')
null2.grid(row=8, column=0)


# 買賣點
BuySelllabel = tk.Label(app, text='根據使用指標決定買賣時機點')
BuySelllabel.grid(row=9, column=1, sticky=tk.W, columnspan=4)
BuySelllabel2 = tk.Label(app, text='系統會將使用指標進行加總')
BuySelllabel2.grid(row=10, column=1, sticky=tk.W, columnspan=4)
BuySelllabel3 = tk.Label(app, text='勾選1個的範圍為1~-1，2個為2~-2')
BuySelllabel3.grid(row=11, column=1, sticky=tk.W, columnspan=5)


buylimitlabel = tk.Label(app, text='買時機:')
buylimitlabel.grid(row=12, column=1)
def show_value_v(value):
    label_v['text'] = value
buyScale = tk.Scale(app, orient='horizontal', from_=-1, to=1, resolution=0.1, showvalue=False, command=show_value_v)
buyScale.grid(row=12, column=2, columnspan=3)
label_v=tk.Label(app)
label_v.grid(row=12, column=5)

selllimitlabel = tk.Label(app, text='賣時機:')
selllimitlabel.grid(row=13, column=1)
def show_value_v2(value):
    label_v2['text'] = value
sellScale = tk.Scale(app, orient='horizontal', from_=-1, to=1,resolution=0.1, showvalue=False, command=show_value_v2)
sellScale.grid(row=13, column=2, columnspan=3)
label_v2=tk.Label(app)
label_v2.grid(row=13, column=5)

null2 = tk.Label(app, text='')
null2.grid(row=14, column=0)


# 選擇策略種類
Strategylabel = tk.Label(app, text='選擇回測策略')
Strategylabel.grid(row=15, column=1, sticky=tk.W, columnspan=2)

radioValue = tk.IntVar()

rdioOne = tk.Radiobutton(app, text='買入持有 (只需填入買時機)',
                             variable=radioValue, value=0)
rdioTwo = tk.Radiobutton(app, text='短期買賣 (買賣時機皆須填入)',
                             variable=radioValue, value=1)
rdioOne.grid(row=16, column=1, sticky=tk.W, columnspan=4)
rdioTwo.grid(row=17, column=1, sticky=tk.W, columnspan=4)

null3 = tk.Label(app, text='')
null3.grid(row=18, column=0)


# 選擇回測時段
timelabel = tk.Label(app, text='回測時段 (過去x年~今日)')
timelabel.grid(row=19, column=1, sticky=tk.W, columnspan=3)
timebox = tk.Spinbox(app,from_=1,to=5, width=10)
timebox.grid(row=19, column=4, sticky=tk.W, columnspan=3)

null4 = tk.Label(app, text='')
null4.grid(row=20, column=0)


# 查詢按鈕
mybutton = tk.Button(app, text='查詢', command=search)
mybutton.grid(row=21, column=1)

# 查詢結果
lbl_1 = tk.Label(app, bg='yellow', fg='#263238')
lbl_1.grid(row=21, column=2, columnspan=4, sticky=tk.W+tk.E)

buylabel = tk.Label(app)
buylabel.grid(row=22, column=2, columnspan=2)
selllabel = tk.Label(app)
selllabel.grid(row=22, column=4, columnspan=2)

#本次交易紀錄按鈕
buttonRecord = tk.Button(app,
              text="本次查詢交易紀錄",
              command=createRecord)
buttonRecord.grid(row=21, column=6, columnspan=3)



#詳細回測資料查詢
Backtestingbutton = tk.Button(app, text='此股票詳細回測資料查詢', command=Backtesting)
Backtestingbutton.grid(row=23, column=1, columnspan=4)
Backtestinglabel2 = tk.Label(app, text='( 詳細資料查詢約8~10分鐘 )')
Backtestinglabel2.grid(row=23, column=5, columnspan=2, sticky=tk.W)

# 詳細回測資料按鈕
Backtestinglabel = tk.Label(app, text='此股票詳細回測資料')
Backtestinglabel.grid(row=24, column=1, columnspan=3)
buttonExample = tk.Button(app,
              text="持有資料",
              command=createBacktesting)
buttonExample.grid(row=24, column=4, columnspan=2)

buttonExample2 = tk.Button(app,
              text="買賣資料",
              command=createBacktesting2)
buttonExample2.grid(row=24, column=6, columnspan=2)

Backtestinglabel.grid_remove()
buttonExample.grid_remove()
buttonExample2.grid_remove()

null6 = tk.Label(app, text='')
null6.grid(row=25, column=0)


#其他標的回測按鈕
OptionList = classificationList

OptionWord = tk.StringVar(app)
OptionWord.set(OptionList[0])

opt = tk.OptionMenu(app, OptionWord, *OptionList)
opt.grid(row=26, column=1, columnspan=3)

optbutton = tk.Button(app, text='此策略在'+OptionWord.get()+'之投資報酬率', bg='PapayaWhip', command=createOption)
optbutton.grid(row=26, column=4, columnspan=4)

def Change_optWord(*args):
    optbutton['text'] = f'此策略在{OptionWord.get()}之投資報酬率'

OptionWord.trace("w", Change_optWord)



app.mainloop()